# GED ML detector with preprocessing

Run Baligh preprocessing, pass its tokens to the uploaded seq labeler

In [37]:
import json
import os
import sys

# Import from the project root when the notebook runs from this directory.
sys.path.append(os.path.abspath("../../../../"))

from src.services.ged.detectors.ml import MLDetector
from src.services.ged.orchestrator import GEDService
from src.services.ged.schemas import GEDInput
from src.services.preprocessing.orchestrator import preprocess
from src.services.preprocessing.schemas import PreprocessingInput

In [38]:
detector = MLDetector()
service = GEDService(subsystems=[detector])

print(f"Loaded {detector.name} with threshold {detector.threshold:.2f}")

Loaded sequence_labeler with threshold 0.35


In [39]:
def test_ml(text: str, *, show_preprocessing: bool = False, raw: bool = False):
    """Run preprocessing and the ML detector for one sentence."""
    pre_output = preprocess(PreprocessingInput(text=text))
    ged_output = service.process(
        GEDInput(
            text=pre_output.text,
            normalized_text=pre_output.normalized_text,
            tokens=pre_output.tokens,
            morph_features=pre_output.morph_features,
        )
    )

    if show_preprocessing:
        print("Preprocessing output:")
        print(json.dumps(pre_output.model_dump(), ensure_ascii=False, indent=2))

    if raw:
        print(json.dumps(ged_output.model_dump(), ensure_ascii=False, indent=2))

    if not ged_output.errors:
        print("No errors found")

    print(f"Found {len(ged_output.errors)} error(s):")
    for error in ged_output.errors:
        surface = ged_output.text[error.span[0] : error.span[1]]
        print(
            f"- {surface!r}: {error.category.value} "
            f"({error.confidence:.3f}), span={error.span}"
        )

## Smoke


In [40]:
test_ml("هاذا كتاب جميل .")

Found 1 error(s):
- 'هاذا': OT (0.988), span=(0, 4)


In [41]:
test_ml("ذهبت الى المدرسة ثم رجعت إلى البيت.")

Found 1 error(s):
- 'الى': OT (0.984), span=(5, 8)


---

## testing


In [ ]:
text = ".ذهبوا امير الى المدسه وحده"
test_ml(text, show_preprocessing=False)

text = "ذهب المعلمين الي الحدددديقققه"
test_ml(text, show_preprocessing=False)

text = "ذهبت إلى المدرسة ثم رجعت إلى البيت."
test_ml(text)

Found 3 error(s):
- 'امير': OT (0.994), span=(7, 11)
- 'الى': OT (0.996), span=(12, 15)
- 'المدسه': OT (0.565), span=(16, 22)
Found 1 error(s):
- 'الي': SY (0.865), span=(13, 16)
No errors found
Found 0 error(s):


: 